# Structural Continuity Analysis of Battery Tomographies — Generation 1: Archimedean Spiral Model

## Introduction

The internal structure of cylindrical batteries consists of thin, tightly wound layers (current collectors, electrodes, and separators) forming a spiral (often referred to as a "jelly roll"). When evaluating the performance of Machine Learning denoising models on neutron tomography data, simple linear checks are insufficient because the layers are inherently curved.

This notebook implements an analytical pipeline that objectively quantifies the structural continuity of these layers. The core idea is **polar unwrapping**: each circular battery cross-section is transformed into polar coordinates and, by compensating for the spiral's pitch (how fast it expands per turn), the curved layers are flattened into straight horizontal bands. Once flattened, any physical break, discontinuity, or artifact introduced by the denoising process becomes a simple gap in an otherwise continuous horizontal line, which background subtraction, adaptive Otsu thresholding, and connected-components analysis can then detect, measure, and map automatically across the full 3D volume. This gives a robust, quantitative way to compare different ML denoising models on the same underlying structural-continuity metric.

## The Model

This is the **first-generation** pipeline. It assumes each winding follows an **Archimedean spiral with constant pitch**:

r(θ_total) = r0 + k·θ_total,  where k = pitch / (2π)

Here `θ_total` is the angle accumulated over multiple turns (not wrapped to [0°, 360°)), so the radius is assumed to grow linearly and at the *same rate* everywhere, regardless of angle or radius. The pitch itself is measured once per slice (`measure_spiral_pitch`, Phase 1) by sampling a single radial ray, and that one global pitch value is used to "de-spiral" the *entire* slice (`spiral_layer_continuity`, Phase 2). An optional refinement (`track_reference_layer` / `flatten_with_reference_layer`) can additionally follow one real, tracked layer to correct for residual distortion the constant-pitch model misses — but this still assumes that correction applies identically to every layer, not just the one being tracked.

See `Structural_Continuity_Analysis_Gen2_GeometryIndependent.ipynb` for the second-generation pipeline, which removes the constant-pitch assumption entirely.

## Limitations

- **Constant-pitch approximation.** Real winding pitch is not perfectly constant; this model captures only the dominant, average trend (Phase 1). In practice, the true pitch tends to drift further from this average near the axial ends of the cell, which is where unwrapping misalignment and false-positive discontinuities are most likely to appear if this pipeline is run across the full height of a cell.
- **Near-center distortion.** Fixed-pitch unwrapping distorts the innermost windings into diagonal, not horizontal, bands, so results below `MIN_RELIABLE_RADIUS` (300 px on this dataset) are excluded from reporting — genuine defects near the center cannot currently be detected reliably.
- **First-order centering correction only.** `refine_center` (Phase 1) fits a single Fourier harmonic to the outer ring; any higher-order, non-elliptical distortion is left uncorrected unless the optional reference-layer tracking (Phase 2) is used — and that correction itself assumes the distortion is shared identically by every layer, which is a simplification, not a guarantee.
- **Casing/outer-ring contamination.** The bright outer casing is thicker than a genuine winding and is a known source of false positives. It is filtered with an empirically tuned width-ratio threshold (Phase 3) and checked diagnostically (Phase 6), but not eliminated by construction.
- **Point vs. area localization.** Large or irregularly shaped defects do not have a single unambiguous "position." The pipeline reports each defect's full angular span rather than collapsing it to a point (Phase 4).
- **No quantitative validation.** The detections have not been compared with an independent reference set of defects, so they should be treated as diagnostic output rather than as a validated defect count.
- **Empirically tuned constants.** Thresholds such as `MIN_RELIABLE_RADIUS`, `MAX_WIDTH_RATIO_VS_MEDIAN`, the material-zone radii, and the minimum defect areas were calibrated on this dataset's geometry and contrast. They will likely need re-tuning for a different cell, resolution, or imaging modality.
- **Known cosmetic issue.** Marker scaling on the `binarized_annotated` diagnostic output (Phase 5) is currently unreliable; use `binarized_raw` instead until this is fixed.

## Attribution

This codebase was developed by Genar Iturri Bello during a summer internship at the Institut Laue-Langevin (ILL), under the supervision of Dr. Bratislav Lukić. The code of both generations is available at https://github.com/genariturri/winding-detection-algorithms. For inquiries or further information, please reach out via email (genariturri@gmail.com) or connect on LinkedIn (https://www.linkedin.com/in/genar-iturri-bello/) or GitHub (https://github.com/genariturri).

## Phase 1: Setup and Geometry Estimation

### Cell 1: Imports and Global Configuration

Imports every library used in the notebook and defines the two global
path constants (`TIFF_DIR`, `out_dir`) that point to the folder of
denoised slices produced by the Noise2Inverse notebook. All later cells
assume these paths are correct.

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import imageio.v2 as imageio
import tifffile
import pandas as pd

from scipy.ndimage import map_coordinates, gaussian_filter1d, gaussian_filter
from scipy.signal import find_peaks
from skimage.transform import warp_polar
from skimage.filters import threshold_otsu
from skimage.measure import label

# Interactive mode off. In Jupyter / VS Code, matplotlib-inline keeps every
# figure created while a cell is running in memory, even after plt.close(),
# until the cell finishes; the batch loop of Phase 5 creates four figures per
# slice and runs out of memory (MemoryError). Figures meant to be seen are
# displayed explicitly with plt.show().
plt.ioff()

# --- Paths: update these to match your own filesystem ---
TIFF_DIR = "/path/to/your/tiffs_battery_sequential"  # <-- update this to your own denoised-slices folder
out_dir = TIFF_DIR
stack_path = os.path.join(TIFF_DIR, "denoised_stack_cropped_centered.tif")

# Total number of slices in the FULL-resolution stack (used below by the
# spiral-overlay sanity check of Cell 5b).
# This is independent of the size of the "overview" stack processed in
# Phase 5, which may cover a smaller or differently-sampled subset.
NUM_SLICES_TO_SAVE = 960

### Cell 2: Automatic Circle Detection and Per-Slice Geometry

`detect_circle` finds a rough (center, radius) for the battery's outer
boundary using a simple intensity threshold and a robust percentile-based
radius estimate. `compute_slice_geometry` builds the full geometric
description of one slice by chaining three independent measurements:
the rough circle, a sub-pixel center correction (`refine_center`,
Cell 4), and the spiral pitch measured at six angles (`measure_spiral_pitch`,
Cell 3). Because this function re-derives everything from scratch for
every slice it is given, the batch pipeline (Phase 5) never assumes that
two slices share the same geometry.

In [ ]:
def detect_circle(raw_image):
    """
    Rough estimate of (center_y, center_x, radius) for the battery's
    outer boundary.

    Approach:
    - Gaussian-blur the image (sigma=5) to suppress fine-grained texture
      noise before thresholding — this is a low-pass filter in the
      spatial-frequency sense, removing structure smaller than ~5 px
      while preserving the macroscopic circular shape.
    - Threshold at 10% of the image's own dynamic range (not an
      absolute intensity value), since normalization can vary
      slice to slice.
    - Center = centroid (mean y, mean x) of the thresholded mask.
      This is the maximum-likelihood center estimator under the
      assumption that the mask is roughly symmetric about the true
      center.
    - Radius = 75th percentile (NOT the maximum) of the distances from
      that centroid to every mask pixel. The percentile is used instead
      of the max because a single noisy outlier pixel far from the
      center would otherwise blow up the radius estimate; the 75th
      percentile is a robust statistic that ignores the outer 25% tail
      of the distance distribution.
    """
    smoothed = gaussian_filter(raw_image.astype("float32"), sigma=5)
    threshold = raw_image.min() + 0.10 * (raw_image.max() - raw_image.min())
    mask = smoothed > threshold

    yy, xx = np.where(mask)
    if len(yy) < 100:
        h, w = raw_image.shape
        print("WARNING: circle detection fell back to image centre.")
        return float(h // 2), float(w // 2), float(min(h, w) // 2)

    center_y = float(np.mean(yy))
    center_x = float(np.mean(xx))
    distances = np.sqrt((yy - center_y) ** 2 + (xx - center_x) ** 2)
    return center_y, center_x, float(np.percentile(distances, 75))


def compute_slice_geometry(image, search_margin=100, verbose=False):
    """
    Full geometric characterization of a single slice: center, radius,
    and spiral pitch — computed independently for THIS slice, never
    reused from another one.

    Returns (center_yx, radius, pitch), or None if the pitch could not
    be measured at any angle.
    """
    # Step 1: rough circle (center + radius)
    center_y, center_x, radius = detect_circle(image)

    # Step 2: sub-pixel center correction, searching for the bright
    # ring within [radius - search_margin, radius]
    r_min = max(0, int(radius) - search_margin)
    r_max = int(radius)
    try:
        center = refine_center(image, (center_y, center_x), r_min, r_max, show_plot=False)
    except Exception as e:
        if verbose:
            print(f"  refine_center failed ({e}); using unrefined center")
        center = (center_y, center_x)

    # Step 3: spiral pitch, measured at 6 angles and combined via median
    # (see measure_spiral_pitch, Cell 3, for why the median is used)
    pitches = []
    for angle in [0, 60, 120, 180, 240, 300]:
        pitch, _, _ = measure_spiral_pitch(image, center, int(radius), angle_degrees=angle, show_plot=False)
        if pitch is not None:
            pitches.append(pitch)

    if len(pitches) == 0:
        return None

    final_pitch = float(np.median(pitches))
    return center, radius, final_pitch

### Cell 3: Spiral Pitch Measurement

Before the spiral can be unwrapped, we need to know how fast its radius
grows per full turn (its "pitch"). `measure_spiral_pitch` samples the
image along a single straight ray from the center outward, detects the
intensity peaks corresponding to successive windings crossed by that
ray, and returns the median spacing between consecutive peaks.

**Underlying model:** the code assumes the winding follows an
Archimedean spiral with constant pitch, `r(θ_total) = r0 + k·θ_total`,
where `θ_total` is the angle accumulated over multiple turns (not
wrapped to [0, 2π)) and `k = pitch / (2π)`. This means the model treats
the radius increase per turn as constant everywhere — independent of
angle and independent of radius. This is a first-order approximation;
Phase 2 discusses where it breaks down (near the center) and how it is
partially compensated (`track_reference_layer`, Cell 5).

In [ ]:
def measure_spiral_pitch(slice_2d, center_yx, max_radius, angle_degrees=0,
                          min_distance_px=15, smoothing_sigma=3, show_plot=True):
    """
    Estimates the spiral pitch (radius increase per full turn) from a
    single radial intensity profile.

    Sampling: parametrizes a straight ray x(r) = cx + r*cos(theta),
    y(r) = cy + r*sin(theta), and reads the image along it with
    scipy.ndimage.map_coordinates(order=1) — i.e. bilinear interpolation,
    since (x(r), y(r)) almost never falls exactly on an integer pixel.

    Peak detection: the raw profile is first smoothed with a 1D Gaussian
    filter (sigma=3) to remove fine texture noise, then scipy.signal.find_peaks
    locates local intensity maxima, each corresponding to a ray crossing
    one winding of the spiral. min_distance_px enforces a minimum spacing
    between detected peaks so that residual noise cannot be mistaken for
    an extra winding.

    Robust pitch estimate: pitch = median(distances between consecutive
    peaks). The median, not the mean, is used because it is insensitive
    to occasional spurious or missed peaks (which would otherwise produce
    one anomalously small or exactly double-sized gap in the distance list).
    """
    cy, cx = center_yx
    theta = np.deg2rad(angle_degrees)

    radii = np.arange(max_radius)
    ys = cy + radii * np.sin(theta)
    xs = cx + radii * np.cos(theta)

    h, w = slice_2d.shape
    valid = (ys >= 0) & (ys < h) & (xs >= 0) & (xs < w)
    radii, ys, xs = radii[valid], ys[valid], xs[valid]

    raw_profile = map_coordinates(slice_2d, [ys, xs], order=1)
    profile = gaussian_filter1d(raw_profile, sigma=smoothing_sigma)

    peaks, _ = find_peaks(profile, distance=min_distance_px)

    if len(peaks) < 2:
        print("WARNING: fewer than 2 peaks detected; try another angle "
              "or adjust min_distance_px/smoothing_sigma.")
        return None, profile, peaks

    distances = np.diff(radii[peaks])
    estimated_pitch = float(np.median(distances))

    if show_plot:
        plt.figure(figsize=(10, 3))
        plt.plot(radii, raw_profile, alpha=0.4, label="Raw profile")
        plt.plot(radii, profile, label="Smoothed profile")
        plt.plot(radii[peaks], profile[peaks], "rx", label="Detected peaks")
        plt.xlabel("Radius (px)")
        plt.ylabel("Intensity")
        plt.title(f"Angle {angle_degrees} deg - estimated pitch: {estimated_pitch:.1f} px/turn")
        plt.legend()
        plt.show()

    print(f"Distances between consecutive peaks: {distances}")
    print(f"Spiral pitch (median): {estimated_pitch:.2f} px per full turn")

    return estimated_pitch, profile, peaks

### Cell 4: Sub-Pixel Center Refinement

`refine_center` corrects small errors in the assumed center by tracking
the radius of the brightest ring (the outer casing / current collector)
across 360 degrees.

**Derivation of the fitted model:** if the true center is offset from
the assumed center by a small vector (δy, δx) with magnitude δ, and the
true ring radius is R, then for δ << R a first-order Taylor expansion
of the law of cosines gives:

    r_measured(θ) ≈ R + δx·cos(θ) + δy·sin(θ)

This is a linear model in (R, δx, δy), fitted to the 720 measured radii
by ordinary least squares (`np.linalg.lstsq`). Because the system has
720 equations for only 3 unknowns, it is heavily over-determined, which
makes the fit robust to noise in any individual angular measurement.
This correction only captures the first Fourier harmonic of the
centering error — any higher-order (non-elliptical) distortion is left
uncorrected here and is partially addressed later by
`track_reference_layer` (Cell 5).

In [ ]:
def refine_center(slice_2d, center_yx, search_radius_min, search_radius_max,
                   n_angles=720, show_plot=True):
    """
    Sub-pixel center correction via least-squares fit of a first-order
    harmonic to the measured ring radius (see markdown above for the
    derivation).
    """
    cy0, cx0 = center_yx
    theta = np.linspace(0, 2 * np.pi, n_angles, endpoint=False)
    peak_radii = np.zeros(n_angles)

    for i, th in enumerate(theta):
        radii = np.arange(search_radius_min, search_radius_max)
        ys = cy0 + radii * np.sin(th)
        xs = cx0 + radii * np.cos(th)
        profile = map_coordinates(slice_2d, [ys, xs], order=1)
        peak_radii[i] = radii[np.argmax(profile)]

    # Design matrix for r(theta) = R0 + dx*cos(theta) + dy*sin(theta)
    design_matrix = np.stack([np.ones_like(theta), np.cos(theta), np.sin(theta)], axis=1)
    coefficients, _, _, _ = np.linalg.lstsq(design_matrix, peak_radii, rcond=None)
    R0, dx, dy = coefficients

    corrected_center = (cy0 + dy, cx0 + dx)

    if show_plot:
        plt.figure(figsize=(10, 3))
        plt.plot(np.rad2deg(theta), peak_radii, label="Measured ring radius")
        plt.plot(np.rad2deg(theta), design_matrix @ coefficients, label="Fit (smooth curve)")
        plt.xlabel("Angle (degrees)")
        plt.ylabel("Ring radius (px)")
        plt.title(f"Center correction: dx={dx:.2f}, dy={dy:.2f} px")
        plt.legend()
        plt.show()

    return corrected_center

## Phase 2: Polar Unwrapping

### Cell 5: The Core Unwrapping Function

`spiral_layer_continuity` transforms a circular slice into a
representation where every healthy winding appears as a clean
horizontal band. Four mathematical steps:

**1. Polar warp.** `skimage.transform.warp_polar` resamples the image
onto a (angle, radius) grid: for output pixel (i, j), it reads the
original image at x = cx + j·cos(θ_i), y = cy + j·sin(θ_i), with
θ_i = i·(2π/n_angles). With `n_angles=1440`, each row covers 0.25°.

**2. Spiral-pitch correction.** Because the winding's radius grows with
accumulated angle (Cell 3's Archimedean model), each angular row is
shifted radially by `radial_shift(θ) = pitch·θ / (2π)` via linear
interpolation (`np.interp`). This "subtracts" the expected radial drift,
stacking every turn on top of the others.

**3. (Optional) Reference-layer flattening.** `track_reference_layer`
follows the true intensity peak of one chosen layer, angle by angle —
capturing whatever residual distortion actually exists, with no assumed
functional form. `flatten_with_reference_layer` then applies that same
correction to the whole image. This assumes the residual distortion is
a rigid-body effect shared by all layers, not something that varies with
radius — a simplification, not a guarantee.

**4. Background subtraction and adaptive thresholding.** A wide
Gaussian blur along the radius axis (`sigma=15`) estimates the local
brightness trend (e.g. the bright outer ring's gradual falloff); the
residual is what is left after subtracting it. Otsu's threshold is then
computed independently in overlapping 40-px radial windows (sampled
every 10 px and linearly interpolated in between) rather than once
globally, because contrast is not uniform with radius: a single global
threshold would be dominated by whichever radius has the widest
intensity spread.

In [ ]:
def track_reference_layer(unwrapped_polar, radius_guess, search_window=15):
    """
    For every angle, finds the radius of the brightest pixel within a
    narrow window around `radius_guess`. Unlike refine_center (which
    only fits a first-order harmonic), this follows the ACTUAL shape of
    the residual distortion for one specific, real layer.
    """
    r0 = max(0, int(radius_guess - search_window))
    r1 = int(radius_guess + search_window)
    local_window = unwrapped_polar[:, r0:r1]
    peak_offset = np.argmax(local_window, axis=1)
    return (r0 + peak_offset).astype(float)


def flatten_with_reference_layer(unwrapped_polar, tracked_radius):
    """
    Shifts every angular row vertically by the deviation of the tracked
    reference layer from its own mean. Assumes this deviation applies
    equally to every layer, not just the tracked one.
    """
    n_angles, n_radii = unwrapped_polar.shape
    correction = tracked_radius - tracked_radius.mean()
    radii = np.arange(n_radii)
    flattened = np.zeros_like(unwrapped_polar)
    for i in range(n_angles):
        flattened[i, :] = np.interp(radii + correction[i], radii, unwrapped_polar[i, :], left=0, right=0)
    return flattened


def spiral_layer_continuity(slice_2d, center_yx, max_radius, spiral_pitch,
                             min_material_radius=190, max_material_radius=630,
                             n_angles=1440, background_sigma=15,
                             threshold_window=40, threshold_step=10,
                             reference_radius=None, reference_window=15):
    """
    Full polar-unwrapping pipeline for one slice. Returns:
        unwrapped_polar    - the desprialed (angle, radius) image
        residual           - unwrapped_polar minus its local background
        binary             - residual thresholded with a radius-dependent
                              adaptive Otsu threshold
        presence_fraction  - fraction of angles with material present,
                              per radius (diagnostic only)
        (min_material_radius, max_material_radius) - passed through unchanged
    """
    # Step 1: polar warp
    polar = warp_polar(slice_2d, center=center_yx, radius=max_radius,
                        output_shape=(n_angles, max_radius))

    # Step 2: subtract the expected radial drift of the Archimedean spiral
    theta = np.linspace(0, 2 * np.pi, n_angles, endpoint=False)
    radial_shift = (spiral_pitch * theta) / (2 * np.pi)

    unwrapped_polar = np.zeros_like(polar)
    radii = np.arange(max_radius)
    for i in range(n_angles):
        source_radii = radii + radial_shift[i]
        unwrapped_polar[i, :] = np.interp(source_radii, radii, polar[i, :], left=0, right=0)

    # Step 3 (optional): flatten residual distortion using a real tracked layer
    if reference_radius is not None:
        tracked = track_reference_layer(unwrapped_polar, reference_radius, reference_window)
        unwrapped_polar = flatten_with_reference_layer(unwrapped_polar, tracked)

    # Step 4a: background subtraction (low-pass filter along the radius axis only)
    background = gaussian_filter1d(unwrapped_polar, sigma=background_sigma, axis=1)
    residual = unwrapped_polar - background

    # Step 4b: adaptive (radius-dependent) Otsu threshold
    sampled_radii = list(range(0, max_radius, threshold_step))
    sampled_thresholds = []
    for c in sampled_radii:
        c0 = max(0, c - threshold_window // 2)
        c1 = min(max_radius, c + threshold_window // 2)
        window_data = residual[:, c0:c1]
        if window_data.max() > window_data.min():
            local_threshold = threshold_otsu(window_data)
        else:
            # Flat window (e.g. outside the sample): no contrast to
            # threshold, fall back to a neutral value.
            local_threshold = window_data.mean()
        sampled_thresholds.append(local_threshold)

    threshold_profile = np.interp(np.arange(max_radius), sampled_radii, sampled_thresholds)
    binary = residual > threshold_profile[np.newaxis, :]

    # Cyclic padding so the 0/360-degree seam is handled correctly by
    # any downstream connected-component analysis
    pad = n_angles // 8
    cyclic_binary = np.concatenate([binary[-pad:, :], binary, binary[:pad, :]], axis=0)
    presence_fraction = cyclic_binary.mean(axis=0)

    return unwrapped_polar, residual, binary, presence_fraction, (min_material_radius, max_material_radius)

In [ ]:
def draw_detected_spiral(image, center_yx, radius_max, spiral_pitch, slice_idx, color='lime', linewidth=1.2):
    """
    Sanity check: overlays the spiral curve that the CURRENT model assumes
    (r = k * theta_total, an Archimedean spiral with constant pitch) directly
    on top of the raw circular tomography image.

    If the model is correct, this green line should sit exactly on top of
    the same physical feature (the same bright/dark band) continuously,
    from near the center all the way out to the edge, without ever
    'jumping' to a neighbouring band.
    """
    k = spiral_pitch / (2 * np.pi)
    n_turns = radius_max / spiral_pitch
    theta_total = np.linspace(0, 2 * np.pi * n_turns, int(2000 * n_turns))
    r = k * theta_total

    xs = center_yx[1] + r * np.cos(theta_total)
    ys = center_yx[0] + r * np.sin(theta_total)

    fig, ax = plt.subplots(figsize=(9, 9))
    ax.imshow(image, cmap="gray")
    ax.plot(xs, ys, color=color, linewidth=linewidth, alpha=0.85)
    ax.plot(center_yx[1], center_yx[0], marker='+', color='red', markersize=12, markeredgewidth=2)
    ax.set_xlim(center_yx[1] - radius_max * 1.05, center_yx[1] + radius_max * 1.05)
    ax.set_ylim(center_yx[0] + radius_max * 1.05, center_yx[0] - radius_max * 1.05)
    ax.set_title(f"Slice {slice_idx} — detected spiral overlay (pitch={spiral_pitch:.1f} px/turn)")
    ax.axis('off')
    plt.show()  # displayed below the cell; not saved to disk


# --- Sanity check on 10 slices, spread across the volume ---
slices_to_check = list(range(0, NUM_SLICES_TO_SAVE, 96))[:10]  # 10 equally spaced slices: 0, 96, 192, ..., 864

with tifffile.TiffFile(stack_path) as tif_check:
    for idx in slices_to_check:
        test_image = tif_check.pages[idx].asarray().astype(np.float32)

        test_geometry = compute_slice_geometry(test_image)
        if test_geometry is None:
            print(f"Slice {idx}: geometry could not be computed, skipping.")
            continue

        test_center, test_radius, test_pitch = test_geometry
        draw_detected_spiral(test_image, test_center, test_radius, test_pitch, idx)

### Cell 5b: Sanity Check — Model Spiral vs. Real Windings

`draw_detected_spiral` (above) draws the spiral `r = k·θ_total` starting at the centre, so its position relative to the windings is arbitrary. This cell anchors the model spiral on a real winding and measures how far the real windings depart from it:

1. **Geometry, exactly as in the batch pipeline.** `compute_slice_geometry` gives the refined centre, the radius and the median pitch `p` that the unwrapping uses.
2. **Anchoring.** On the reference ray (`SANITY_REFERENCE_ANGLE_DEG`), the bright winding closest to the middle of the analysed zone is located, and the model spiral `r = r_anchor + (p/2π)·θ_total` is drawn through it, with the same angle convention as `warp_polar` (`x = cx + r·cos φ`, `y = cy + r·sin φ`). If the constant-pitch model were exact, this curve would stay on the same winding from the inner to the outer radius.
3. **Real winding through the anchor.** Around every point of the model spiral, the radial profile (±1.5 periods) is projected onto `exp(−2πiu/P)` (lock-in detection). The phase gives the offset of the nearest bright winding, and continuity from the anchor point identifies the winding that passes through it. Its offset `d(θ)` from the model is split into a secular part (pitch error) and a first harmonic (residual centring error).
4. **Local pitch of the real windings.** (i) The radial period of the windings at every radius, from all angles at once (pulse-pair estimator), and (ii) the radial distance between successive turns of the tracked winding. Neither depends on the model; for a constant-pitch spiral both would be flat and equal to `p`.

Panels: (a) model spiral on the slice, (b) zoom where the model is farthest from the real winding, (c) image sampled along the model spiral, (d) profile of the reference ray with the peaks used for the pitch and the crossings predicted by the model, (e) local pitch versus radius, (f) centre refinement (`refine_center`: radius of the brightest ring versus angle and its first-harmonic fit). The composite figure is displayed below the cell; each panel is also saved as a PNG file, together with two CSV tables, in `spiral_sanity_check/`. Change `SANITY_SLICES` to check other slices.


In [ ]:
# =====================================================================
# Cell 5b: Sanity check - model spiral vs. real windings
# =====================================================================
# For every slice in SANITY_SLICES this cell:
#   1. computes the geometry exactly as the batch pipeline does
#      (compute_slice_geometry: refined centre, radius, median pitch);
#   2. anchors the model spiral r = r_anchor + k*theta_total (k = pitch/2pi)
#      on a real bright winding of the reference ray, near mid-radius;
#   3. draws that spiral on the tomographic slice, with the same angle
#      convention as warp_polar (x = cx + r cos(phi), y = cy + r sin(phi));
#   4. follows the real winding that passes through the anchor point, turn
#      after turn, and measures how far the model spiral is from it;
#   5. measures the local pitch of the real windings at every radius;
#   6. shows the centre refinement (ring radius versus angle and its fit).
# If the constant-pitch model were exact, the model spiral would stay on the
# anchored winding from the inner to the outer radius.
#
# Outputs (in SANITY_OUT_DIR, one set per slice):
#   slice_XXX_sanity_summary.png  all panels together
#   slice_XXX_overlay.png         (a) model spiral on the slice, (b) zoom
#   slice_XXX_strip.png           (c) image along the model spiral
#   slice_XXX_profile.png         (d) reference-ray profile and peaks
#   slice_XXX_local_pitch.png     (e) local pitch of the real winding
#   slice_XXX_centre.png          (f) centre refinement of refine_center
#   spiral_sanity_summary.csv     one row per slice
#   spiral_sanity_local_pitch.csv local pitch per turn, per slice
# =====================================================================
import contextlib
import io
import warnings
from matplotlib import patheffects
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle

# --- Configuration ---------------------------------------------------
SANITY_SLICES = [100, 192, 320, 480, 640, 768, 830]  # slice indices in stack_path (inside the stack)
SANITY_REFERENCE_ANGLE_DEG = 0.0   # ray on which the model spiral is anchored
SANITY_R_INNER = 190               # material zone, as min_material_radius (Cell 5)
SANITY_R_OUTER = 630               # material zone, as max_material_radius (Cell 5)
SANITY_MIN_RELIABLE_RADIUS = 300   # same value as MIN_RELIABLE_RADIUS (Cell 8)
SANITY_STEP_DEG = 0.5              # angular step along the spiral
SANITY_OUT_DIR = os.path.join(TIFF_DIR, "spiral_sanity_check")
os.makedirs(SANITY_OUT_DIR, exist_ok=True)

MODEL_COLOR = "#ff8c00"    # model spiral (orange)
TRACK_COLOR = "#00bfff"    # tracked real winding (sky blue)
ANCHOR_COLOR = "#ff00ff"   # anchor point (magenta)


# --- Helpers -----------------------------------------------------------
def _quiet(func, *args, **kwargs):
    """Runs func discarding what it prints (measure_spiral_pitch prints
    the peak spacings at every call)."""
    with contextlib.redirect_stdout(io.StringIO()):
        return func(*args, **kwargs)


def sample_polar(image, center_yx, phi, radii, half_width_deg=0.25, n_rays=3):
    """Bilinear samples at x = cx + r cos(phi), y = cy + r sin(phi), the
    convention of warp_polar and measure_spiral_pitch. phi has shape (n,),
    radii shape (n, m). Each sample is averaged over n_rays rays spread
    over +-half_width_deg to reduce noise."""
    cy, cx = center_yx
    out = np.zeros(radii.shape, dtype=np.float64)
    for offset in np.deg2rad(np.linspace(-half_width_deg, half_width_deg, n_rays)):
        ang = (phi + offset)[:, None]
        ys = cy + radii * np.sin(ang)
        xs = cx + radii * np.cos(ang)
        out += map_coordinates(image, [ys.ravel(), xs.ravel()], order=1,
                               mode="nearest").reshape(radii.shape)
    return out / n_rays


def find_anchor_radius(image, center_yx, period, ref_angle_rad, r_lo, r_hi):
    """Radius of the bright winding closest to the middle of [r_lo, r_hi]
    on the reference ray. The profile is smoothed at the scale of the winding
    period (sigma = period/6), so that the two maxima of one double-sided
    electrode count as a single winding."""
    radii = np.arange(int(r_lo), int(r_hi), dtype=float)
    phis = ref_angle_rad + np.deg2rad(np.linspace(-1.0, 1.0, 5))
    prof = sample_polar(image, center_yx, phis, np.tile(radii, (len(phis), 1))).mean(axis=0)
    prof = gaussian_filter1d(prof, sigma=max(2.0, period / 6.0))
    peaks, _ = find_peaks(prof, distance=max(3, int(0.5 * period)))
    target = 0.5 * (r_lo + r_hi)
    if len(peaks) == 0:
        return float(target)
    j = peaks[np.argmin(np.abs(radii[peaks] - target))]
    shift = 0.0
    if 0 < j < len(prof) - 1:                      # parabolic sub-pixel refinement
        y0, y1, y2 = prof[j - 1], prof[j], prof[j + 1]
        denom = y0 - 2.0 * y1 + y2
        if denom < 0:
            shift = 0.5 * (y0 - y2) / denom
    return float(radii[j] + shift)


def model_spiral(center_yx, r_anchor, pitch, ref_angle_rad, r_inner, r_outer, step_rad):
    """Spiral assumed by the unwrapping of Cell 5, r = r_anchor + k*theta_total
    with k = pitch/(2 pi), theta_total measured from the reference ray in the
    sense of warp_polar. Returns theta_total, r, phi, x, y and the index of
    the anchor point (theta_total = 0)."""
    k = pitch / (2 * np.pi)
    i_min = int(np.ceil((r_inner - r_anchor) / k / step_rad))
    i_max = int(np.floor((r_outer - r_anchor) / k / step_rad))
    theta_total = np.arange(i_min, i_max + 1) * step_rad
    r = r_anchor + k * theta_total
    phi = ref_angle_rad + theta_total
    x = center_yx[1] + r * np.cos(phi)
    y = center_yx[0] + r * np.sin(phi)
    return theta_total, r, phi, x, y, -i_min


def winding_period(residual_turn, pitch):
    """Dominant radial period of the windings (px): peak of the radial power
    spectrum of the background-subtracted polar image, averaged over all
    angles, searched between 0.4 and 3 times the pitch of the algorithm (and
    moved to the fundamental if the peak found is the second harmonic).
    It is the reference period of the phase measurement below."""
    n_r = residual_turn.shape[1]
    data = (residual_turn - residual_turn.mean(axis=1, keepdims=True)) * np.hanning(n_r)
    power = np.mean(np.abs(np.fft.rfft(data, n=8 * n_r, axis=1)) ** 2, axis=0)
    freqs = np.fft.rfftfreq(8 * n_r)
    band = (freqs > 1.0 / min(3.0 * pitch, 150.0)) & (freqs < 1.0 / max(0.4 * pitch, 8.0))
    f_peak = freqs[band][np.argmax(power[band])]
    # If half that frequency also carries power, the peak was the second
    # harmonic of a winding with two maxima per period: keep the fundamental
    near = (freqs > 0.85 * f_peak / 2) & (freqs < 1.15 * f_peak / 2) & (freqs > 1.0 / 150.0)
    if near.any() and power[near].max() > 0.15 * power[np.argmin(np.abs(freqs - f_peak))]:
        f_peak = freqs[near][np.argmax(power[near])]
    return float(1.0 / f_peak)


def local_radial_period(residual_turn, r_cols, period):
    """Local radial period of the windings as a function of radius, from all
    angles at once (pulse-pair estimator): the profile of every angle is
    demodulated at the mean period, low-pass filtered, and the average phase
    advance per pixel gives the local frequency. For an ideal spiral this
    radial period is the local pitch, independently of any model. Returns the
    period (px) for every radius of r_cols and its coherence (0-1)."""
    z = residual_turn * np.exp(-2j * np.pi * r_cols / period)[None, :]
    z = gaussian_filter1d(z.real, period / 2, axis=1) + 1j * gaussian_filter1d(z.imag, period / 2, axis=1)
    pair = np.sum(z[:, 1:] * np.conj(z[:, :-1]), axis=0)
    coherence = np.abs(pair) / (np.sum(np.abs(z[:, 1:]) * np.abs(z[:, :-1]), axis=0) + 1e-12)
    freq = gaussian_filter1d(1.0 / period + np.angle(pair) / (2 * np.pi), period / 2)
    freq = np.append(freq, freq[-1])
    coherence = np.append(coherence, coherence[-1])
    return 1.0 / np.clip(freq, 1.0 / (2.0 * period), 1.0 / (0.5 * period)), coherence


def analyse_spiral_model(image, idx):
    """Geometry, anchored model spiral and real winding for one slice."""
    geometry = _quiet(compute_slice_geometry, image)
    if geometry is None:
        return None
    center, radius, pitch = geometry
    ref = np.deg2rad(SANITY_REFERENCE_ANGLE_DEG)
    step = np.deg2rad(SANITY_STEP_DEG)

    # Pitch estimate of every ray and peaks of the reference ray, exactly as
    # measure_spiral_pitch computes them for compute_slice_geometry
    ray_pitches = {}
    for angle in [0, 60, 120, 180, 240, 300]:
        p_ray, _, _ = _quiet(measure_spiral_pitch, image, center, int(radius),
                             angle_degrees=angle, show_plot=False)
        ray_pitches[angle] = p_ray
    _, ref_profile, ref_peaks = _quiet(measure_spiral_pitch, image, center, int(radius),
                                       angle_degrees=SANITY_REFERENCE_ANGLE_DEG, show_plot=False)
    ref_radii = np.arange(int(radius))
    ys = center[0] + ref_radii * np.sin(ref)
    xs = center[1] + ref_radii * np.cos(ref)
    inside = (ys >= 0) & (ys < image.shape[0]) & (xs >= 0) & (xs < image.shape[1])
    ref_radii = ref_radii[inside]
    ref_raw = map_coordinates(image, [ys[inside], xs[inside]], order=1)

    # Centre refinement as done by compute_slice_geometry (refine_center),
    # recomputed to keep the measured ring radii for panel (f)
    cy0, cx0, rough_radius = detect_circle(image)
    win = np.arange(max(0, int(rough_radius) - 100), int(rough_radius))
    th_c = np.linspace(0, 2 * np.pi, 720, endpoint=False)
    profiles_c = map_coordinates(image, [(cy0 + win[None, :] * np.sin(th_c)[:, None]).ravel(),
                                         (cx0 + win[None, :] * np.cos(th_c)[:, None]).ravel()],
                                 order=1).reshape(len(th_c), len(win))
    ring = win[np.argmax(profiles_c, axis=1)].astype(float)
    A_c = np.stack([np.ones_like(th_c), np.cos(th_c), np.sin(th_c)], axis=1)
    coef_c = np.linalg.lstsq(A_c, ring, rcond=None)[0]
    centre_fit = {"theta": th_c, "ring": ring, "fit": A_c @ coef_c, "dx": coef_c[1], "dy": coef_c[2],
                  "r_min": float(win[0]), "r_max": float(win[-1]),
                  "clipped": (ring <= win[0]) | (ring >= win[-1])}

    # Polar image of one full turn starting at the reference ray, restricted
    # to the material zone, with the background removed as in Cell 5. Along
    # the spiral the same polar image is read again at every turn.
    r_inner = float(SANITY_R_INNER)
    r_outer = float(min(SANITY_R_OUTER, radius - 2))
    n_turn = int(round(360.0 / SANITY_STEP_DEG))
    phi_turn = ref + np.arange(n_turn) * step
    r_cols = np.arange(int(np.floor(r_inner)), int(np.ceil(r_outer)) + 1, dtype=float)
    polar_turn = sample_polar(image, center, phi_turn, np.tile(r_cols, (n_turn, 1)))
    residual_turn = polar_turn - gaussian_filter1d(polar_turn, sigma=15, axis=1)
    P = winding_period(residual_turn, pitch)
    if r_outer - r_inner < 4 * P:
        print(f"Slice {idx}: material zone too narrow for the check, skipping.")
        return None

    # Model spiral of the algorithm (its own pitch), anchored on a real
    # bright winding of the reference ray near mid-radius
    r_lo = max(SANITY_MIN_RELIABLE_RADIUS, r_inner)
    if r_outer - r_lo < 2 * P:
        r_lo = r_inner
    r_anchor = find_anchor_radius(image, center, P, ref, r_lo, r_outer)
    theta_total, r_model, phi, x_model, y_model, i0 = model_spiral(
        center, r_anchor, pitch, ref, r_inner, r_outer, step)
    n = len(theta_total)
    row_of = (np.arange(n) - i0) % n_turn

    def along_spiral(offsets):
        """Residual image at radius r_model + offset for every point of the
        model spiral (NaN outside the material zone)."""
        cols = (r_model[:, None] + offsets[None, :]) - r_cols[0]
        rows = np.broadcast_to(row_of[:, None], cols.shape)
        return map_coordinates(residual_turn, [rows.ravel(), cols.ravel()], order=1,
                               mode="constant", cval=np.nan).reshape(cols.shape)

    # Local radial period of the windings at every radius (all angles)
    period_r, coherence_r = local_radial_period(residual_turn, r_cols, P)
    trusted_r = (coherence_r > 0.8) & (r_cols >= r_inner + P) & (r_cols <= r_outer - 2 * P)

    # Position of the real winding relative to the model by lock-in detection:
    # around every point of the model spiral the radial profile (+-1.5 periods)
    # is projected on exp(-2 pi i u / P_i), P_i being the local period; the
    # phase gives the offset of the nearest bright winding within half a
    # period, and continuity from the anchor point decides which winding it
    # is (the one that passes through the anchor).
    P_rows = np.interp(r_model, r_cols, period_r)
    W = int(round(1.5 * P))
    uu = np.arange(-W, W + 1, dtype=float)
    local_profiles = along_spiral(uu)
    valid_fraction = np.mean(np.isfinite(local_profiles), axis=1)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)     # rows entirely outside the zone
        local_profiles = np.nan_to_num(local_profiles - np.nanmean(local_profiles, axis=1, keepdims=True))
    kernel = np.exp(-2j * np.pi * uu[None, :] / P_rows[:, None]) * np.hanning(len(uu))[None, :]
    zc = np.sum(local_profiles * kernel, axis=1)
    smooth = max(1.0, 2.0 / SANITY_STEP_DEG)              # ~2 degrees along the spiral
    zc = gaussian_filter1d(zc.real, smooth) + 1j * gaussian_filter1d(zc.imag, smooth)
    amplitude = np.abs(zc)
    usable = valid_fraction > 0.9
    level = np.median(amplitude[usable]) if usable.any() else np.median(amplitude)
    reliable = usable & (amplitude > 0.3 * level)
    nearest = -P_rows * np.angle(zc) / (2 * np.pi)       # nearest winding, within +-P/2
    d = np.full(n, np.nan)
    good = np.flatnonzero(reliable)
    if good.size > 10:
        d_good = np.empty(good.size)
        j0 = int(np.argmin(np.abs(good - i0)))
        d_good[j0] = nearest[good[j0]]
        for order in (range(j0 + 1, good.size), range(j0 - 1, -1, -1)):
            prev = d_good[j0]
            for j in order:
                g = good[j]
                k = np.round((prev - nearest[g]) / P_rows[g])
                # spacing of the k windings in between, taken at their midpoint
                p_mid = np.interp(r_model[g] + nearest[g] + 0.5 * k * P_rows[g], r_cols, period_r)
                k = np.round((prev - nearest[g]) / p_mid)
                d_good[j] = nearest[g] + k * p_mid
                prev = d_good[j]
        d = np.interp(np.arange(n), good, d_good)
    r_track = r_model + d                          # real winding through the anchor
    reliable &= (r_track >= r_inner + 0.5 * P) & (r_track <= r_outer - 0.5 * P)

    # Residual image along the model spiral, for display (rows = position
    # along the spiral, columns = radial offset from the model)
    half = int(round(6 * pitch))
    u = np.arange(-half, half + 1)
    residual = along_spiral(u.astype(float))

    # Decomposition of the offset: secular part (pitch) + first harmonic (centre)
    fit = {}
    if reliable.sum() > 50:
        A = np.column_stack([np.ones_like(theta_total), theta_total, theta_total ** 2,
                             np.cos(phi), np.sin(phi)])
        coef, *_ = np.linalg.lstsq(A[reliable], d[reliable], rcond=None)
        fit = {"slope_px_per_turn": 2 * np.pi * coef[1],
               "centre_offset_dx": coef[3], "centre_offset_dy": coef[4]}

    # Local pitch of the real winding: radial distance to the same winding one
    # turn later (this cancels the centring error, which repeats every turn),
    # averaged turn by turn
    local = []
    if n > n_turn:
        p_loc = r_track[n_turn:] - r_track[:-n_turn]
        both = reliable[n_turn:] & reliable[:-n_turn]
        r_mid = 0.5 * (r_track[n_turn:] + r_track[:-n_turn])
        turn_id = np.arange(len(p_loc)) // n_turn
        for t in np.unique(turn_id):
            sel = (turn_id == t) & both
            r_turn = r_mid[sel].mean() if sel.any() else np.nan
            if sel.sum() > 0.5 * n_turn and r_inner + P <= r_turn <= r_outer - 2 * P:
                local.append({"slice": idx, "turn": int(t), "radius_px": float(r_mid[sel].mean()),
                              "local_pitch_px": float(p_loc[sel].mean()),
                              "local_pitch_std_px": float(p_loc[sel].std())})

    # Turns from the anchor after which the model is more than half a period
    # away from the real winding, i.e. it already lies on a neighbouring one
    def turns_to_half_period(sl):
        bad = reliable[sl] & (np.abs(d[sl]) > 0.5 * P)
        return float(abs(theta_total[sl][np.argmax(bad)]) / (2 * np.pi)) if bad.any() else np.nan

    zone = reliable & (r_model >= SANITY_MIN_RELIABLE_RADIUS)
    return {
        "idx": idx, "image": image, "center": center, "radius": radius, "pitch": pitch,
        "period": P, "r_cols": r_cols, "period_r": period_r, "trusted_r": trusted_r,
        "ray_pitches": ray_pitches, "ref_radii": ref_radii, "ref_raw": ref_raw,
        "ref_profile": ref_profile[:len(ref_radii)], "ref_peaks": ref_peaks[ref_peaks < len(ref_radii)],
        "r_inner": r_inner, "r_outer": r_outer, "r_anchor": r_anchor, "i0": i0,
        "theta_total": theta_total, "r_model": r_model, "phi": phi,
        "x_model": x_model, "y_model": y_model, "u": u, "residual": residual,
        "d": d, "r_track": r_track, "reliable": reliable, "fit": fit, "local": local,
        "turns_out": turns_to_half_period(slice(i0, None)),
        "turns_in": turns_to_half_period(slice(i0, None, -1)),
        "max_abs_offset": float(np.abs(d[zone]).max()) if zone.any() else np.nan,
        "reliable_fraction": float(reliable.mean()), "centre_fit": centre_fit,
    }


# --- Plotting -------------------------------------------------------------
def _display_range(res):
    """Grey-level range of the windings (annulus of the material zone)."""
    img = res["image"]
    cy, cx = res["center"]
    yy, xx = np.ogrid[:img.shape[0], :img.shape[1]]
    rr2 = (yy - cy) ** 2 + (xx - cx) ** 2
    zone = (rr2 > res["r_inner"] ** 2) & (rr2 < (res["r_outer"] - 0.5 * res["period"]) ** 2)
    return np.percentile(img[zone], [0.5, 99.5])


def _masked(values, keep):
    out = np.asarray(values, dtype=float).copy()
    out[~keep] = np.nan
    return out


def _zoom_point(res):
    """Point of the model spiral where the real winding is farthest from it
    (reliable part, inside the analysed zone); the opposite side of the
    reference ray if nothing reliable is found."""
    sel = res["reliable"] & (res["r_model"] >= SANITY_MIN_RELIABLE_RADIUS) & \
          (res["r_model"] <= res["r_outer"] - res["pitch"])
    if sel.any():
        i = np.flatnonzero(sel)[np.argmax(np.abs(res["d"][sel]))]
    else:
        i = int(np.argmin(np.abs(res["theta_total"] - np.pi)))
    return res["x_model"][i], res["y_model"][i]


def draw_overlay(ax_full, ax_zoom, res):
    img = res["image"]
    cy, cx = res["center"]
    lo, hi = _display_range(res)
    keep = res["reliable"]
    x_track = res["center"][1] + res["r_track"] * np.cos(res["phi"])
    y_track = res["center"][0] + res["r_track"] * np.sin(res["phi"])
    ref = np.deg2rad(SANITY_REFERENCE_ANGLE_DEG)
    xa = cx + res["r_anchor"] * np.cos(ref)
    ya = cy + res["r_anchor"] * np.sin(ref)
    zx, zy = _zoom_point(res)
    hz = max(60.0, 3.0 * res["pitch"])
    th = np.linspace(0, 2 * np.pi, 721)

    for ax, lw in ((ax_full, 0.9), (ax_zoom, 1.8)):
        ax.imshow(img, cmap="gray", vmin=lo, vmax=hi, interpolation="nearest")
        ax.plot(res["x_model"], res["y_model"], color=MODEL_COLOR, lw=lw, label="model spiral")
        ax.plot(_masked(x_track, keep), _masked(y_track, keep), color=TRACK_COLOR, lw=lw * 0.9,
                ls=(0, (4, 2)), label="tracked real winding")
        ax.plot(cx + SANITY_MIN_RELIABLE_RADIUS * np.cos(th), cy + SANITY_MIN_RELIABLE_RADIUS * np.sin(th),
                color="white", lw=0.7, ls=":", label=f"r = {SANITY_MIN_RELIABLE_RADIUS} px")
        ax.plot(xa, ya, marker="*", ms=13 if ax is ax_zoom else 10, color=ANCHOR_COLOR,
                mec="black", mew=0.6, ls="none", label="anchor point")
        ax.set_xticks([])
        ax.set_yticks([])

    R = res["radius"]
    ax_full.plot([cx + res["r_inner"] * np.cos(ref), cx + res["r_outer"] * np.cos(ref)],
                 [cy + res["r_inner"] * np.sin(ref), cy + res["r_outer"] * np.sin(ref)],
                 color=ANCHOR_COLOR, lw=0.8, ls="--")
    ax_full.plot(cx, cy, marker="+", color="red", ms=12, mew=2)
    ax_full.add_patch(Rectangle((zx - hz, zy - hz), 2 * hz, 2 * hz, fill=False, ec="yellow", lw=1.2))
    ax_full.set_xlim(cx - 1.06 * R, cx + 1.06 * R)
    ax_full.set_ylim(cy + 1.06 * R, cy - 1.06 * R)
    ax_full.set_title(f"(a) Slice {res['idx']}: model spiral, pitch {res['pitch']:.1f} px/turn", fontsize=11)

    ax_zoom.set_xlim(zx - hz, zx + hz)
    ax_zoom.set_ylim(zy + hz, zy - hz)
    outline = [patheffects.withStroke(linewidth=4, foreground="black")]
    ax_zoom.plot([zx + hz - 50 - 0.08 * hz, zx + hz - 0.08 * hz], [zy + 0.88 * hz] * 2, color="white", lw=3,
                 path_effects=outline)
    ax_zoom.text(zx + hz - 25 - 0.08 * hz, zy + 0.82 * hz, "50 px", color="white", ha="center",
                 va="bottom", fontsize=9, path_effects=outline)
    ax_zoom.set_title("(b) Zoom on the yellow box", fontsize=11)
    ax_zoom.legend(loc="upper left", fontsize=8, framealpha=0.85)


def draw_strip(ax, res):
    turns = res["theta_total"] / (2 * np.pi)
    u = res["u"]
    p = res["pitch"]
    d_rel = np.abs(res["d"][res["reliable"]])
    height = float(np.clip((d_rel.max() if d_rel.size else 0.0) + 0.6 * p, 2.0 * p, 6.0 * p))
    show = np.abs(u) <= height
    resid = res["residual"][:, show]
    lo, hi = np.nanpercentile(resid, [1, 99])
    cmap = plt.get_cmap("gray").copy()
    cmap.set_bad("#d9d9d9")                  # outside the material zone
    ax.imshow(np.ma.masked_invalid(resid.T), aspect="auto", origin="lower", cmap=cmap, vmin=lo, vmax=hi,
              extent=[turns[0], turns[-1], u[show][0], u[show][-1]], interpolation="nearest")
    ax.axhline(0, color=MODEL_COLOR, lw=1.3, ls="--", label="model spiral (offset 0)")
    d_show = _masked(res["d"], res["reliable"] & (np.abs(res["d"]) <= height))
    ax.plot(turns, d_show, color=TRACK_COLOR, lw=1.2, label="tracked real winding")
    ax.axvline(0, color=ANCHOR_COLOR, lw=1.0, ls=":", label="anchor point")
    for k in (-1, 1):
        ax.axhline(k * p / 2, color="white", lw=0.6, ls=":")
    ax.set_xlabel("Turns from the anchor point (along the model spiral)")
    ax.set_ylabel("Radial offset from\nthe model (px)")
    ax.set_title("(c) Image sampled along the model spiral: a correct model keeps the "
                 "anchored winding on the dashed line", fontsize=11)
    sec = ax.secondary_xaxis("top", functions=(lambda t: res["r_anchor"] + p * t,
                                               lambda r: (r - res["r_anchor"]) / p))
    sec.set_xlabel("Radius of the model spiral (px)", fontsize=9)
    ax.legend(loc="lower right", fontsize=8, framealpha=0.85)


def draw_profile(ax, res):
    r = res["ref_radii"]
    ax.plot(r, res["ref_raw"], color="0.7", lw=0.7, label="raw profile")
    ax.plot(r, res["ref_profile"], color="0.15", lw=1.0, label="smoothed profile ($\\sigma$ = 3 px)")
    pk = res["ref_peaks"]
    ax.plot(r[pk], res["ref_profile"][pk], "x", color="#d62728", ms=6, mew=1.5,
            label="peaks used for the pitch")
    n_lo = int(np.ceil((res["r_inner"] - res["r_anchor"]) / res["pitch"]))
    n_hi = int(np.floor((res["r_outer"] - res["r_anchor"]) / res["pitch"]))
    for n in range(n_lo, n_hi + 1):
        ax.axvline(res["r_anchor"] + n * res["pitch"], color=MODEL_COLOR, lw=0.8, ls="--",
                   label="model crossings" if n == n_lo else None)
    ax.axvspan(0, SANITY_MIN_RELIABLE_RADIUS, color="0.92", zorder=0,
               label=f"excluded (r < {SANITY_MIN_RELIABLE_RADIUS} px)")
    ax.set_xlim(0, r[-1])
    ax.set_xlabel("Radius (px)")
    ax.set_ylabel("Intensity")
    ref_p = res["ray_pitches"].get(int(SANITY_REFERENCE_ANGLE_DEG))
    ref_txt = f"{ref_p:.1f}" if ref_p is not None else "n/a"
    ax.set_title(f"(d) Reference ray ({SANITY_REFERENCE_ANGLE_DEG:.0f} deg): pitch of this ray "
                 f"{ref_txt} px, slice pitch {res['pitch']:.1f} px", fontsize=11)
    ax.legend(loc="upper right", fontsize=7, ncol=2, framealpha=0.85)


def draw_local_pitch(ax, res):
    rc, pr, ok = res["r_cols"], res["period_r"], res["trusted_r"]
    ax.plot(rc, _masked(pr, ok), color="0.25", lw=1.6,
            label="radial period of the windings\n(all angles)")
    if res["local"]:
        rr = [t["radius_px"] for t in res["local"]]
        pp = [t["local_pitch_px"] for t in res["local"]]
        ss = [t["local_pitch_std_px"] for t in res["local"]]
        ax.errorbar(rr, pp, yerr=ss, fmt="o", color=TRACK_COLOR, mec="black", mew=0.5, ms=6,
                    capsize=3, label="tracked winding, turn by turn")
    ax.axhline(res["pitch"], color=MODEL_COLOR, lw=1.5, ls="--",
               label=f"model pitch ({res['pitch']:.1f} px)")
    ax.axvspan(0, SANITY_MIN_RELIABLE_RADIUS, color="0.92", zorder=0)
    ax.set_xlim(res["r_inner"] - 10, res["r_outer"] + 10)
    ax.set_xlabel("Radius (px)")
    ax.set_ylabel("Local pitch (px per turn)")
    ax.set_title("(e) Local pitch of the real windings", fontsize=11)
    ax.legend(loc="best", fontsize=8, framealpha=0.85)


def draw_centre_fit(ax, res):
    cf = res["centre_fit"]
    deg = np.rad2deg(cf["theta"])
    ax.plot(deg, cf["ring"], color="0.35", lw=0.9, label="radius of the brightest ring")
    if cf["clipped"].any():
        ax.plot(deg[cf["clipped"]], cf["ring"][cf["clipped"]], ".", color="#d62728", ms=3,
                label="at the limit of the search window")
    ax.plot(deg, cf["fit"], color=MODEL_COLOR, lw=1.8, label="first-harmonic fit")
    for r_lim in (cf["r_min"], cf["r_max"]):
        ax.axhline(r_lim, color="0.6", lw=0.8, ls=":")
    ax.set_xlim(0, 360)
    ax.set_xticks(range(0, 361, 60))
    ax.set_xlabel("Angle (degrees)")
    ax.set_ylabel("Ring radius (px)")
    ax.set_title(f"(f) Centre refinement: dx = {cf['dx']:.1f} px, dy = {cf['dy']:.1f} px; "
                 f"{100 * cf['clipped'].mean():.0f}% of the angles at the window limits", fontsize=11)
    ax.legend(loc="best", fontsize=8, framealpha=0.85)


def plot_sanity_check(res, out_dir):
    tag = f"slice_{res['idx']:03d}"
    # All panels together (displayed in the notebook)
    fig = plt.figure(figsize=(13, 18.5))
    gs = GridSpec(4, 1, figure=fig, height_ratios=[6.2, 2.9, 3.0, 2.6], hspace=0.42)
    top = gs[0].subgridspec(1, 2, wspace=0.06)
    middle = gs[2].subgridspec(1, 2, wspace=0.28, width_ratios=[1.35, 1])
    draw_overlay(fig.add_subplot(top[0]), fig.add_subplot(top[1]), res)
    draw_strip(fig.add_subplot(gs[1]), res)
    draw_profile(fig.add_subplot(middle[0]), res)
    draw_local_pitch(fig.add_subplot(middle[1]), res)
    draw_centre_fit(fig.add_subplot(gs[3]), res)
    fig.savefig(os.path.join(out_dir, f"{tag}_sanity_summary.png"), dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    # Each panel on its own, for the report
    fig, (a, b) = plt.subplots(1, 2, figsize=(13, 6.6), gridspec_kw={"wspace": 0.05})
    draw_overlay(a, b, res)
    fig.savefig(os.path.join(out_dir, f"{tag}_overlay.png"), dpi=200, bbox_inches="tight")
    plt.close(fig)
    for name, func, size in (("strip", draw_strip, (12, 3.6)), ("profile", draw_profile, (11, 3.6)),
                             ("local_pitch", draw_local_pitch, (6.5, 4.2)), ("centre", draw_centre_fit, (11, 3.4))):
        fig, ax = plt.subplots(figsize=size)
        func(ax, res)
        fig.savefig(os.path.join(out_dir, f"{tag}_{name}.png"), dpi=200, bbox_inches="tight")
        plt.close(fig)


# --- Run ------------------------------------------------------------------
summary_rows, local_rows = [], []
with tifffile.TiffFile(stack_path) as tif_sanity:
    n_pages = len(tif_sanity.pages)
    outside = [s for s in SANITY_SLICES if not 0 <= s < n_pages]
    if outside:
        print(f"Slices {outside} are outside the stack ({n_pages} slices) and are skipped.")
    for idx in [s for s in SANITY_SLICES if 0 <= s < n_pages]:
        image = tif_sanity.pages[idx].asarray().astype(np.float32)
        try:
            res = analyse_spiral_model(image, idx)
            if res is None:
                print(f"Slice {idx}: could not be analysed, skipping.")
                continue
            plot_sanity_check(res, SANITY_OUT_DIR)
        except Exception as error:
            print(f"Slice {idx}: sanity check failed ({error}), skipping.")
            continue
        fit = res["fit"]
        lp = [t["local_pitch_px"] for t in res["local"]]
        zone_r = res["trusted_r"] & (res["r_cols"] >= SANITY_MIN_RELIABLE_RADIUS)
        summary_rows.append({
            "slice": idx,
            "center_y": res["center"][0], "center_x": res["center"][1], "radius": res["radius"],
            "pitch_model_px": res["pitch"],
            "radial_period_px": res["period"],
            **{f"pitch_ray_{a}deg": v for a, v in res["ray_pitches"].items()},
            "anchor_radius_px": res["r_anchor"],
            "reliable_fraction": res["reliable_fraction"],
            "max_abs_offset_px": res["max_abs_offset"],
            "turns_to_half_period_outward": res["turns_out"],
            "turns_to_half_period_inward": res["turns_in"],
            "drift_px_per_turn_at_anchor": fit.get("slope_px_per_turn", np.nan),
            "centre_correction_dx_px": res["centre_fit"]["dx"],
            "centre_correction_dy_px": res["centre_fit"]["dy"],
            "ring_at_window_limits_fraction": float(res["centre_fit"]["clipped"].mean()),
            "residual_centre_offset_px": float(np.hypot(fit.get("centre_offset_dx", np.nan),
                                                        fit.get("centre_offset_dy", np.nan))),
            "local_pitch_min_px": min(lp) if lp else np.nan,
            "local_pitch_max_px": max(lp) if lp else np.nan,
            "radial_period_min_px": float(np.min(res["period_r"][zone_r])) if zone_r.any() else np.nan,
            "radial_period_max_px": float(np.max(res["period_r"][zone_r])) if zone_r.any() else np.nan,
        })
        local_rows.extend(res["local"])

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(SANITY_OUT_DIR, "spiral_sanity_summary.csv"), index=False)
pd.DataFrame(local_rows).to_csv(os.path.join(SANITY_OUT_DIR, "spiral_sanity_local_pitch.csv"), index=False)
if len(summary_df):
    print(summary_df[["slice", "pitch_model_px", "radial_period_px", "anchor_radius_px",
                      "max_abs_offset_px", "turns_to_half_period_outward", "turns_to_half_period_inward",
                      "drift_px_per_turn_at_anchor", "residual_centre_offset_px",
                      "local_pitch_min_px", "local_pitch_max_px",
                      "radial_period_min_px", "radial_period_max_px"]].round(2).to_string(index=False))
print(f"\nFigures and tables saved in: {SANITY_OUT_DIR}")

## Phase 3: Discontinuity Detection

### Cell 6: Connected-Components Analysis

`detect_discontinuities` labels every connected region in the binarized,
unwrapped image (`skimage.measure.label`, 8-connectivity — each pixel
connects to all 8 neighbours, including diagonals, so a diagonally
sloped band from imperfect unwrapping is not artificially fragmented).

For every region:
- **`angular_extent`** / **`continuous`**: a region spanning nearly the
  full 1440-sample circle (accounting for the cyclic padding) is a
  healthy, unbroken layer.
- **`gap_angle_rad`**: for broken regions, the **circular mean** of the
  missing angles — `atan2(mean(sin θ), mean(cos θ))` over the angles
  where the region is absent. A circular mean is required because plain
  averaging is undefined across the 0/360-degree seam (e.g. averaging
  359° and 1° should give 0°, not 180°).
- **`centroid_angle_rad` / `centroid_radius_px`**: the circular mean and
  the linear mean taken over ALL pixels of the region, not just the
  missing ones — more robust for large, irregular blobs where
  `gap_angle_rad` would land in the middle of the healthy arc instead of
  at the actual anomaly.
- **`start_angle_rad` / `end_angle_rad`**: the true angular span the
  region occupies — the lossless description of "where" the defect is,
  used later to paint it as a region rather than compress it to a point.
- **`radial_half_width_px`**: the true measured radial thickness of the
  fragment, used later both for drawing and for filtering out
  abnormally thick structures (e.g. the outer casing) that are not
  genuine electrode layers.

In [ ]:
def detect_discontinuities(binary, n_angles, min_area=50, edge_margin=3):
    """
    Connected-component analysis of a binarized, unwrapped image. See
    the markdown above for what each returned field means and why.
    """
    pad = n_angles // 4
    cyclic_binary = np.concatenate([binary[-pad:, :], binary, binary[:pad, :]], axis=0)

    labels = label(cyclic_binary, connectivity=2)
    n_regions = labels.max()
    n_radii = binary.shape[1]

    results = []
    for label_idx in range(1, n_regions + 1):
        mask = labels == label_idx
        area = mask.sum()
        if area < min_area:
            continue

        present_cols = np.where(mask.any(axis=0))[0]
        touches_edge = (present_cols.min() < edge_margin) or \
                       (present_cols.max() > n_radii - 1 - edge_margin)
        if touches_edge:
            # A region touching the crop's own radial edge is likely a
            # crop artefact, not a genuine defect - discard it.
            continue

        present_rows = np.where(mask.any(axis=1))[0]
        angular_extent = present_rows.max() - present_rows.min()
        is_continuous = angular_extent >= (n_angles - pad * 0.5)
        mean_radius = present_cols.mean()
        radial_half_width_px = float(present_cols.max() - present_cols.min()) / 2.0

        start_angle_rad = ((present_rows.min() - pad) % n_angles) * (2 * np.pi / n_angles)
        end_angle_rad = ((present_rows.max() - pad) % n_angles) * (2 * np.pi / n_angles)

        # Circular mean of the MISSING angles (only meaningful for broken regions)
        gap_angle_rad = None
        if not is_continuous:
            original_mask = mask[pad: pad + n_angles, :]
            present_angles = np.where(original_mask.any(axis=1))[0]
            missing_angles = list(set(range(n_angles)) - set(present_angles))
            if len(missing_angles) > 0:
                rads = np.array(missing_angles) * (2 * np.pi / n_angles)
                gap_angle_rad = np.arctan2(np.mean(np.sin(rads)), np.mean(np.cos(rads)))
                if gap_angle_rad < 0:
                    gap_angle_rad += 2 * np.pi

        # Circular mean over ALL pixels of the region (robust centroid,
        # used instead of gap_angle_rad for large/irregular blobs)
        rows_px, cols_px = np.where(mask)
        original_rows = (rows_px - pad) % n_angles
        angles_px = original_rows * (2 * np.pi / n_angles)
        centroid_angle_rad = np.arctan2(np.mean(np.sin(angles_px)), np.mean(np.cos(angles_px))) % (2 * np.pi)
        centroid_radius_px = float(cols_px.mean())

        results.append({
            "label": label_idx,
            "mean_radius_px": mean_radius,
            "area_px": int(area),
            "angular_extent": int(angular_extent),
            "continuous": bool(is_continuous),
            "gap_angle_rad": gap_angle_rad,
            "centroid_radius_px": centroid_radius_px,
            "centroid_angle_rad": centroid_angle_rad,
            "start_angle_rad": start_angle_rad,
            "end_angle_rad": end_angle_rad,
            "radial_half_width_px": radial_half_width_px,
        })
    return results

### Cell 7: Duplicate Removal (Cyclic-Seam Cleanup)

The cyclic padding used above can split a single real region into two
labels right at the seam. `remove_duplicate_detections` merges any two
detections whose radius is nearly identical (within `radius_tolerance`
pixels), keeping only the larger-area one — a simple but effective fix
for this specific artefact.

In [ ]:
def remove_duplicate_detections(detections, radius_tolerance=3):
    """
    If two detections have almost identical radius, they are almost
    certainly the same physical region, split into two labels by the
    cyclic padding at the 0/360-degree seam. Keep only the larger one.
    """
    sorted_detections = sorted(detections, key=lambda x: x["mean_radius_px"])
    deduplicated = []

    for detection in sorted_detections:
        duplicate_index = None
        for i, existing in enumerate(deduplicated):
            if abs(existing["mean_radius_px"] - detection["mean_radius_px"]) <= radius_tolerance:
                duplicate_index = i
                break

        if duplicate_index is None:
            deduplicated.append(detection)
        elif detection["area_px"] > deduplicated[duplicate_index]["area_px"]:
            deduplicated[duplicate_index] = detection

    return deduplicated

### Cell 8: Combined Per-Slice Analysis — Fissures and Fusions

`analyze_slice_defects` is the complete per-slice pipeline used by the
batch loop (Phase 5). It runs the unwrapping once, then detects two
distinct physical defect types on the same binarized image:

- **Fissures**: a layer breaks — material is missing where it should be
  continuous. Detected by running `detect_discontinuities` directly on
  the binary image.
- **Fusions**: two neighbouring layers merge — the normal air gap
  between them disappears. Detected by running the SAME detector on the
  **inverted** binary image (`~binary`), where "material" now means
  "gap"; a gap fragment that does not span the full circle marks a
  point where material has bridged across it.

Two filters are then applied, calibrated per slice and per defect type
(never with a single global constant, since both healthy-layer and
healthy-gap thickness vary somewhat from slice to slice):

1. **Reliable-radius filter** (`MIN_RELIABLE_RADIUS = 300`): the
   fixed-pitch unwrapping model is known (from direct visual inspection)
   to distort the innermost windings into diagonal bands rather than
   horizontal ones, generating spurious detections. Everything below
   this radius is excluded.
2. **Relative-thickness filter** (`MAX_WIDTH_RATIO_VS_MEDIAN = 1.5`):
   for each defect type separately, the median radial thickness of that
   type's own healthy (continuous) fragments is computed, and any
   fragment — healthy or broken — thicker than 1.5x that median is
   discarded. This removes structures that are not genuine electrode
   layers (most notably the bright outer casing/current-collector ring,
   which is measurably thicker than any individual winding). The 1.5x
   ratio was chosen empirically: across several manually inspected
   slices, the observed ratio between a normal layer and the casing
   ring ranged from about 1.5x to 3x, so 1.5 is the most conservative
   value that still separates the two populations.

In [ ]:
MIN_RELIABLE_RADIUS = 300
MAX_WIDTH_RATIO_VS_MEDIAN = 1.5


def analyze_slice_defects(image, center_yx, max_radius, spiral_pitch,
                           min_area_fissure=50, min_area_fusion=400):
    """
    Full per-slice defect analysis. Returns (filtered_defects,
    (report_radius_start, report_radius_end), binary_image).
    """
    _, _, binary, _, (rmin, rmax) = spiral_layer_continuity(
        image, center_yx=center_yx, max_radius=max_radius, spiral_pitch=spiral_pitch
    )

    report_radius_start = max(rmin, MIN_RELIABLE_RADIUS)

    # --- Fissures: a layer breaks ---
    fissures = detect_discontinuities(binary, n_angles=binary.shape[0], min_area=min_area_fissure)
    for r in fissures:
        r["defect_type"] = "fissure"
    fissures = remove_duplicate_detections(fissures)

    # --- Fusions: two layers merge (their normal gap disappears) ---
    gap_mask = ~binary
    fusions = detect_discontinuities(gap_mask, n_angles=gap_mask.shape[0], min_area=min_area_fusion)
    for r in fusions:
        r["defect_type"] = "fusion"
    fusions = remove_duplicate_detections(fusions)

    def compute_width_limit(fragments):
        """Median thickness of this type's own healthy fragments, times
        MAX_WIDTH_RATIO_VS_MEDIAN. Falls back to 'no limit' if there are
        too few healthy fragments in this slice to calibrate against."""
        healthy_widths = [r["radial_half_width_px"] for r in fragments if r["continuous"]]
        if len(healthy_widths) >= 3:
            return float(np.median(healthy_widths)) * MAX_WIDTH_RATIO_VS_MEDIAN
        return float("inf")

    fissure_width_limit = compute_width_limit(fissures)
    fusion_width_limit = compute_width_limit(fusions)

    filtered_fissures = [r for r in fissures if r["radial_half_width_px"] <= fissure_width_limit]
    filtered_fusions = [r for r in fusions if r["radial_half_width_px"] <= fusion_width_limit]

    all_defects = filtered_fissures + filtered_fusions
    filtered_defects = [r for r in all_defects if report_radius_start <= r["mean_radius_px"] <= rmax]

    return filtered_defects, (report_radius_start, rmax), binary

## Phase 4: Visualization Utilities

### Cell 9: Drawing Helpers

Three small utilities used by the batch loop (Phase 5):

- **`set_axes_in_degrees`**: relabels a binarized-image x-axis from raw
  angular samples (0-1440) to degrees (0-360), since `sample = degree *
  n_angles / 360`.
- **`draw_polar_ruler`**: overlays radius rings and angle ticks (in the
  same coordinate convention the algorithm itself uses) directly on the
  circular tomography image, plus a pixel scale bar, so a defect can be
  cross-checked visually without ambiguity.
- **`draw_defect_ribbon`**: paints a defect as the real angular arc it
  occupies — `r(θ) = base_radius + pitch·θ/(2π) ± half_width` for
  `θ ∈ [start_angle, end_angle]` — instead of collapsing it to a single
  point. Angular resolution scales with the defect's own angular size
  (~2 points per degree, up to 400 points), so a 2-degree defect is not
  drawn with the same coarseness as a 180-degree one. This directly
  reflects the fact that a defect is a region, not a point: for large or
  irregular defects, no single position (centroid, circular mean of the
  missing angles or midpoint of the angular limits) represents where the
  defect is without ambiguity.

In [ ]:
def set_axes_in_degrees(ax, total_angle_samples):
    """Relabels the x-axis of a binarized-image plot from angular
    samples to degrees, in 30-degree steps."""
    degree_ticks = list(range(0, 361, 30))
    tick_samples = [d * total_angle_samples / 360 for d in degree_ticks]
    ax.set_xticks(tick_samples)
    ax.set_xticklabels([f"{d} deg" for d in degree_ticks])
    ax.set_xlabel("Angle (degrees)")
    ax.set_ylabel("Radius (px)")


def draw_polar_ruler(ax, center_yx, radius, margin, tick_length):
    """Draws magenta angle ticks + radius rings + a pixel scale bar on a
    circular tomography image, using the same (center, radius) the
    detection algorithm itself uses."""
    max_ruler_radius = radius * margin

    for ring_radius in range(100, int(radius), 100):
        guide_circle = plt.Circle((center_yx[1], center_yx[0]), ring_radius,
                                   color="magenta", fill=False, linewidth=0.6,
                                   linestyle="--", alpha=0.6)
        ax.add_patch(guide_circle)
        lx = center_yx[1] + ring_radius * np.cos(np.deg2rad(45))
        ly = center_yx[0] + ring_radius * np.sin(np.deg2rad(45))
        ax.text(lx, ly, f"{ring_radius}px", color="magenta", fontsize=6, ha="center", va="center", alpha=0.9)

    for angle_deg in range(0, 360, 30):
        angle_rad = np.deg2rad(angle_deg)
        tick_end_radius = radius * (1 + tick_length)
        text_radius = radius * (1 + tick_length * 1.8)
        x_edge = center_yx[1] + radius * np.cos(angle_rad)
        y_edge = center_yx[0] + radius * np.sin(angle_rad)
        x_tick = center_yx[1] + tick_end_radius * np.cos(angle_rad)
        y_tick = center_yx[0] + tick_end_radius * np.sin(angle_rad)
        x_text = center_yx[1] + text_radius * np.cos(angle_rad)
        y_text = center_yx[0] + text_radius * np.sin(angle_rad)
        ax.plot([x_edge, x_tick], [y_edge, y_tick], color="magenta", linewidth=1)
        ax.text(x_text, y_text, f"{angle_deg} deg", color="magenta", fontsize=6, ha="center", va="center")

    ax.set_xlim(center_yx[1] - max_ruler_radius, center_yx[1] + max_ruler_radius)
    ax.set_ylim(center_yx[0] + max_ruler_radius, center_yx[0] - max_ruler_radius)

    scale_px = 100
    x0_scale = ax.get_xlim()[1] - scale_px * 1.3
    y0_scale = ax.get_ylim()[0] - (ax.get_ylim()[0] - ax.get_ylim()[1]) * 0.05
    ax.plot([x0_scale, x0_scale + scale_px], [y0_scale, y0_scale], color="white", linewidth=3)
    ax.text(x0_scale + scale_px / 2, y0_scale + (ax.get_ylim()[0] - ax.get_ylim()[1]) * 0.03,
            f"{scale_px} px", color="white", fontsize=9, ha="center", va="top")


def draw_defect_ribbon(ax, center_yx, base_radius, spiral_pitch, start_angle, end_angle,
                        half_width_px, color, alpha=0.55, max_points=400):
    """Paints a defect as the real angular span it occupies, following
    the spiral's curvature, with its own measured radial thickness and
    an angular resolution that scales with its angular size."""
    a0, a1 = start_angle, end_angle
    if a1 < a0:
        a1 += 2 * np.pi

    angular_span_deg = np.degrees(a1 - a0)
    n_points = max(8, min(max_points, int(angular_span_deg * 2)))  # ~2 points per degree
    angles = np.linspace(a0, a1, n_points) % (2 * np.pi)

    true_radii = base_radius + (spiral_pitch * angles) / (2 * np.pi)
    r_outer = true_radii + half_width_px
    r_inner = true_radii - half_width_px

    xs_outer = center_yx[1] + r_outer * np.cos(angles)
    ys_outer = center_yx[0] + r_outer * np.sin(angles)
    xs_inner = center_yx[1] + r_inner[::-1] * np.cos(angles[::-1])  # radii reversed with the angles
    ys_inner = center_yx[0] + r_inner[::-1] * np.sin(angles[::-1])

    xs = np.concatenate([xs_outer, xs_inner])
    ys = np.concatenate([ys_outer, ys_inner])
    ax.fill(xs, ys, color=color, alpha=alpha, linewidth=0)

## Phase 5: Full-Volume Batch Analysis

### Cell 10: Processing the Full Overview Stack

Loops through every slice of the overview stack (its slice count is
printed as it loads). For each
slice: computes fresh geometry (Phase 1), runs `analyze_slice_defects`
(Phase 3), and saves four diagnostic images per slice:

- `fissures_only/` and `fusions_only/`: the circular tomography image
  with only that defect type painted as a real region (Phase 4), each
  with its own area-based colorbar.
- `binarized_raw/`: the plain unwrapped/binarized image, with no
  markers — the most direct representation of what the algorithm itself
  analyzed.
- `binarized_annotated/`: the same image with markers overlaid.
  **Known limitation, not yet resolved:** the marker scaling on this
  output is currently unreliable and should not be used for
  verification until fixed; `binarized_raw` is the trustworthy
  alternative in the meantime.

Per-slice geometry is logged to `slice_geometry_log.csv` for later
auditing (e.g. checking whether the spiral pitch is unexpectedly
unstable in a particular range of slices).

In [ ]:
import contextlib
import gc
import io
import matplotlib.colors as mcolors

# Keep memory flat during the loop: no interactive figure tracking (see Cell 1),
# all figures closed after every slice, garbage collected regularly.
plt.ioff()

MIN_DEFECT_AREA = 50
MAX_DEFECT_AREA = 500
RULER_MARGIN = 1.08
DEGREE_TICK_LENGTH = 0.04

# stack_path is already defined above (Phase 1, Cell 1)
binarized_raw_dir = os.path.join(TIFF_DIR, "binarized_raw")
binarized_annotated_dir = os.path.join(TIFF_DIR, "binarized_annotated")
fissures_only_dir = os.path.join(TIFF_DIR, "fissures_only")
fusions_only_dir = os.path.join(TIFF_DIR, "fusions_only")
for folder in (binarized_raw_dir, binarized_annotated_dir, fissures_only_dir, fusions_only_dir):
    os.makedirs(folder, exist_ok=True)

full_global_summary = []
slice_detections = {}
geometry_log = []

cmap_fissure = plt.get_cmap("YlOrRd")
cmap_fusion = plt.get_cmap("Blues")
norm = mcolors.Normalize(vmin=MIN_DEFECT_AREA, vmax=MAX_DEFECT_AREA)

with tifffile.TiffFile(stack_path) as tif:
    total_slices = len(tif.pages)
    print(f"Overview volume loaded: {total_slices} slices.")
    print(f"Outputs will be saved in:\n  {fissures_only_dir}\n  {fusions_only_dir}\n"
          f"  {binarized_raw_dir}\n  {binarized_annotated_dir}\n")

    t0 = time.time()

    for idx in range(total_slices):
        image = tif.pages[idx].asarray().astype(np.float32)

        with contextlib.redirect_stdout(io.StringIO()):   # pitch is logged in the CSV below
            geometry = compute_slice_geometry(image)
        if geometry is None:
            print(f"  slice {idx}: geometry could not be computed, skipping.")
            continue
        slice_center, slice_radius, slice_pitch = geometry
        geometry_log.append({
            "slice": idx, "center_y": slice_center[0], "center_x": slice_center[1],
            "radius": slice_radius, "pitch": slice_pitch,
        })

        defects, (rmin_slice, rmax_slice), binary_image = analyze_slice_defects(
            image, center_yx=slice_center, max_radius=int(slice_radius), spiral_pitch=slice_pitch,
        )
        slice_detections[idx] = defects

        broken_defects = [r for r in defects if not r["continuous"]]
        n_fissures = sum(1 for r in broken_defects if r["defect_type"] == "fissure")
        n_fusions = sum(1 for r in broken_defects if r["defect_type"] == "fusion")

        for r in broken_defects:
            full_global_summary.append({
                "overview_slice": idx,
                "radius_px": r["mean_radius_px"],
                "area_px": r["area_px"],
                "angular_extent": r["angular_extent"],
                "defect_type": r["defect_type"],
                "start_angle_deg": float(np.degrees(r["start_angle_rad"])) if r.get("start_angle_rad") is not None else None,
                "end_angle_deg": float(np.degrees(r["end_angle_rad"])) if r.get("end_angle_rad") is not None else None,
            })

        file_name = f"slice_{idx:03d}_fissures{n_fissures}_fusions{n_fusions}.png"

        # --- Output A: fissures only ---
        fig_f, (cax_f, ax_f) = plt.subplots(1, 2, figsize=(10 * RULER_MARGIN, 8 * RULER_MARGIN),
                                             gridspec_kw={"width_ratios": [0.05, 1]})
        ax_f.imshow(image, cmap="gray")
        draw_polar_ruler(ax_f, slice_center, slice_radius, RULER_MARGIN, DEGREE_TICK_LENGTH)

        for r in broken_defects:
            if r["defect_type"] != "fissure" or r.get("start_angle_rad") is None:
                continue
            color = cmap_fissure(norm(r["area_px"]))
            draw_defect_ribbon(ax_f, slice_center, r["mean_radius_px"], slice_pitch,
                                r["start_angle_rad"], r["end_angle_rad"],
                                r.get("radial_half_width_px", 8), color)

        ax_f.axis("off")
        sm_f = plt.cm.ScalarMappable(cmap=cmap_fissure, norm=norm)
        sm_f.set_array([])
        cbar_f = fig_f.colorbar(sm_f, cax=cax_f)
        cbar_f.ax.yaxis.set_ticks_position("left")
        cbar_f.ax.yaxis.set_label_position("left")
        cbar_f.set_label("Fissure Size (Area in px)", rotation=90, labelpad=10)
        fig_f.savefig(os.path.join(fissures_only_dir, file_name), bbox_inches="tight", dpi=150, facecolor="white")
        plt.close(fig_f)

        # --- Output B: fusions only ---
        fig_fu, (ax_fu, cax_fu) = plt.subplots(1, 2, figsize=(10 * RULER_MARGIN, 8 * RULER_MARGIN),
                                                gridspec_kw={"width_ratios": [1, 0.05]})
        ax_fu.imshow(image, cmap="gray")
        draw_polar_ruler(ax_fu, slice_center, slice_radius, RULER_MARGIN, DEGREE_TICK_LENGTH)

        for r in broken_defects:
            if r["defect_type"] != "fusion" or r.get("start_angle_rad") is None:
                continue
            color = cmap_fusion(norm(r["area_px"]))
            draw_defect_ribbon(ax_fu, slice_center, r["mean_radius_px"], slice_pitch,
                                r["start_angle_rad"], r["end_angle_rad"],
                                r.get("radial_half_width_px", 8), color)

        ax_fu.axis("off")
        sm_fu = plt.cm.ScalarMappable(cmap=cmap_fusion, norm=norm)
        sm_fu.set_array([])
        cbar_fu = fig_fu.colorbar(sm_fu, cax=cax_fu)
        cbar_fu.set_label("Fusion Size (Area in px)", rotation=270, labelpad=15)
        fig_fu.savefig(os.path.join(fusions_only_dir, file_name), bbox_inches="tight", dpi=150, facecolor="white")
        plt.close(fig_fu)

        # --- Output C: raw binarized image, no markers ---
        total_angle_samples = binary_image.shape[0]
        fig_raw, ax_raw = plt.subplots(figsize=(14, 6))
        ax_raw.imshow(binary_image[:, rmin_slice:rmax_slice].T, aspect="auto", cmap="gray", origin="lower",
                      extent=[0, total_angle_samples, rmin_slice, rmax_slice])
        set_axes_in_degrees(ax_raw, total_angle_samples)
        fig_raw.savefig(os.path.join(binarized_raw_dir, file_name), bbox_inches="tight", dpi=150, facecolor="white")
        plt.close(fig_raw)

        # --- Output D: binarized image with markers ---
        # KNOWN LIMITATION: marker scaling on this output is currently
        # unreliable; use Output C for verification until this is fixed.
        fig_ann, ax_ann = plt.subplots(figsize=(14, 6))
        ax_ann.imshow(binary_image[:, rmin_slice:rmax_slice].T, aspect="auto", cmap="gray", origin="lower",
                      extent=[0, total_angle_samples, rmin_slice, rmax_slice])
        set_axes_in_degrees(ax_ann, total_angle_samples)

        for r in broken_defects:
            if r.get("start_angle_rad") is None:
                continue
            cmap_used = cmap_fissure if r["defect_type"] == "fissure" else cmap_fusion
            color = cmap_used(norm(r["area_px"]))
            a0_sample = (r["start_angle_rad"] / (2 * np.pi)) * total_angle_samples
            a1_sample = (r["end_angle_rad"] / (2 * np.pi)) * total_angle_samples
            if a1_sample < a0_sample:
                a1_sample += total_angle_samples

            if r["defect_type"] == "fissure":
                ax_ann.plot([a0_sample, a1_sample], [r["mean_radius_px"], r["mean_radius_px"]],
                            color=color, linewidth=4, alpha=0.85)
            else:
                ax_ann.plot(a0_sample % total_angle_samples, r["mean_radius_px"], marker="|",
                            color=color, markersize=16, markeredgewidth=3)
                ax_ann.plot(a1_sample % total_angle_samples, r["mean_radius_px"], marker="|",
                            color=color, markersize=16, markeredgewidth=3)

        fig_ann.savefig(os.path.join(binarized_annotated_dir, file_name), bbox_inches="tight", dpi=150, facecolor="white")
        plt.close(fig_ann)

        plt.close("all")
        if idx % 20 == 0:
            gc.collect()

        if idx > 0 and idx % 10 == 0:
            elapsed = time.time() - t0
            print(f"  Processed and saved {idx}/{total_slices} slices ({elapsed:.1f}s elapsed)", flush=True)

print("\nAnalysis complete.")

geometry_df = pd.DataFrame(geometry_log)
geometry_csv_path = os.path.join(TIFF_DIR, "slice_geometry_log.csv")
geometry_df.to_csv(geometry_csv_path, index=False)
print(f"Geometry log saved to: {geometry_csv_path}")

### Cell 11: Aggregation, Export, and Spatial Mapping

Collects every detected defect across the whole volume into a single
DataFrame, exports it (with its full angular span, in degrees — the
lossless data needed for a 3D reconstruction) to
`discontinuities_summary_overview.csv`, and renders the top-level result
of the notebook: a scatter map of slice index vs. radius, point size
proportional to defect area, colored by defect type. This shows whether
defects are isolated to single slices or extend across a height range,
and whether they cluster at particular radii.

In [ ]:
full_df = pd.DataFrame(full_global_summary)

if len(full_df) == 0:
    print("No discontinuities detected in this overview.")
else:
    csv_path = os.path.join(TIFF_DIR, "discontinuities_summary_overview.csv")
    full_df.to_csv(csv_path, index=False)
    print(f"Summary saved to: {csv_path}")

    plt.figure(figsize=(12, 6))
    for defect_type, color, label in [("fissure", "red", "Fissures (broken layer)"),
                                       ("fusion", "deepskyblue", "Fusions (merged layers)")]:
        subset = full_df[full_df["defect_type"] == defect_type]
        plt.scatter(subset["overview_slice"], subset["radius_px"],
                    s=subset["area_px"] / 50, alpha=0.5, c=color, label=label)

    plt.xlabel(f"Slice index (0 to {total_slices - 1})")
    plt.ylabel("Radius (px)")
    plt.title(f"Discontinuities Map (Overview of {total_slices} slices)")
    plt.legend()
    plt.gca().invert_yaxis()
    plt.show()

    print(f"\nTotal fissures: {(full_df['defect_type']=='fissure').sum()}")
    print(f"Total fusions: {(full_df['defect_type']=='fusion').sum()}")

## Phase 6: Quality Control

### Cell 12: Ring-Contamination Check

`check_ring_contamination` tests whether detected defects
disproportionately cluster in the outermost fraction of the analyzed
radius range — the region closest to the bright casing/current-collector
ring, a known potential source of false positives due to its dominant
contrast. A result below the internal 40% threshold indicates the
bright ring is not systematically driving the defect counts, though it
does not rule out other sources of false positives (e.g. the abnormally
thick-region filter in Cell 8, or the reliable-radius cutoff itself).

In [ ]:
def check_ring_contamination(slice_detections, rmin, rmax, outer_fraction_threshold=0.85):
    """
    outer_fraction_threshold: fraction of [rmin, rmax] beyond which a
    defect is considered 'suspiciously close' to the outer ring.
    0.85 means: the outermost 15% of the analyzed radius range.
    """
    total = 0
    near_ring_count = 0
    radius_threshold = rmin + outer_fraction_threshold * (rmax - rmin)

    for idx, detections in slice_detections.items():
        broken = [d for d in detections if not d["continuous"]]
        for d in broken:
            total += 1
            if d["mean_radius_px"] >= radius_threshold:
                near_ring_count += 1

    if total == 0:
        print("No defects detected to evaluate.")
        return

    fraction = near_ring_count / total
    print(f"Total defects analyzed: {total}")
    print(f"Defects in the outermost {100*(1-outer_fraction_threshold):.0f}% "
          f"(radius >= {radius_threshold:.0f}px): {near_ring_count} ({fraction:.0%})")

    if fraction > 0.4:
        print("\nWARNING: a large fraction of defects cluster near the outer edge.")
        print("This is consistent with contamination from the bright outer ring (false positives).")
    else:
        print("\nOK: defects do not suspiciously cluster near the outer ring.")

### Cell 13: Running the Ring-Contamination Check

Uses the rmin/rmax of a representative slice (these should be similar
across slices, since they depend on the fixed material-zone bounds, not
on the per-slice dynamic geometry).

In [ ]:
check_ring_contamination(slice_detections, rmin=190, rmax=630, outer_fraction_threshold=0.85)